In [1]:
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from tabulate import tabulate
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [2]:
def preprocess_tokens(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    return [w for w in words if w not in ENGLISH_STOP_WORDS]

In [5]:
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

In [6]:
tokenized_data = [preprocess_tokens(doc) for doc in dataset]


In [7]:
model = Word2Vec(sentences=tokenized_data, vector_size=100, window=5, min_count=1)

In [8]:
X = np.array([
    np.mean([model.wv[word] for word in doc], axis=0)
    for doc in tokenized_data
])

In [9]:
km = KMeans(n_clusters=2, random_state=42)
km.fit(X)
y_pred = km.predict(X)

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [10]:
table_data = [["Document", "Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))


Document                                           Cluster
-----------------------------------------------  ---------
I love playing football on the weekends                  0
I enjoy hiking and camping in the mountains              0
I like to read books and watch movies                    1
I prefer playing video games over sports                 0
I love listening to music and going to concerts          1


In [11]:
purity = max(Counter(y_pred).values()) / len(y_pred)
print("Purity:", purity)

Purity: 0.6
